# PB01 — BCI Illiteracy Prediction from Rest EEG

**Obiettivo**: validare la claim di Blankertz et al. — le caratteristiche EEG a riposo predicono la performance nel task di imagined speech.

**Pipeline**:
1. Carica epoche `riposo_NNN.csv` per ogni soggetto
2. Estrae feature spettrali (alpha, beta, SMR power) e di connettività
3. Correla con la `bacc` soggetto-specifica (da W&B / EEG_37)
4. Classificatore binario: BCI-literate vs BCI-illiterate

**Riferimento**: Blankertz et al. 2010 — *Neurophysiological predictor of SMR-based BCI performance*

In [ ]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists()),
    Path().resolve()
)

DATA_ROOT = project_root / 'data' / 'raw_csv' / 'training_set'

SFREQ   = 256
N_CHAN  = 61
N_SAMP  = 384

# Soglia per definire BCI-illiterate (bacc <= threshold = illiterate)
ILLITERACY_THRESHOLD = 0.30  # 30% bacc su 4 classi (chance=25%)

print(f'DATA_ROOT: {DATA_ROOT}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print('Import OK')

In [ ]:
# ============================================================
# CARICA BACC PER SOGGETTO
# Fonte: PB00_compute_subject_bacc.ipynb (CSV locale)
#        oppure W&B se preferisci usare EEG_06 / EEG_13b
# ============================================================

BACC_CSV = project_root / 'data' / 'interim' / 'subject_bacc_pipelineB.csv'

if BACC_CSV.exists():
    _df_bacc = pd.read_csv(BACC_CSV).dropna(subset=['bacc'])
    SUBJ_BACC = dict(zip(_df_bacc['subj_id'].astype(int), _df_bacc['bacc']))
    print(f'Caricati {len(SUBJ_BACC)} soggetti da {BACC_CSV.name}')
    print(f'bacc media: {_df_bacc["bacc"].mean():.3f}  range: [{_df_bacc["bacc"].min():.3f}, {_df_bacc["bacc"].max():.3f}]')
else:
    print('WARNING: subject_bacc_pipelineB.csv non trovato.')
    print('Esegui PB00_compute_subject_bacc.ipynb prima di questo notebook.')
    SUBJ_BACC = {}  # notebook continua senza correlazioni

In [ ]:
# ============================================================
# CARICA EPOCHE RIPOSO PER SOGGETTO
# Media su sessioni → (n_riposo_tot, 61, 384) per soggetto
# ============================================================

def load_rest_epochs(subj_id: int, data_root: Path) -> np.ndarray:
    """Carica tutte le epoche riposo di un soggetto (tutte le sessioni).
    Ritorna array (n_epochs, 61, 384) float32."""
    epochs = []
    for sess_dir in sorted(data_root.glob(f'P{subj_id:03d}_S*')):
        for f in sorted(sess_dir.glob('riposo_*.csv')):
            x = pd.read_csv(f, header=None).values.astype(np.float32)
            if x.shape == (N_CHAN, N_SAMP):
                epochs.append(x)
    return np.array(epochs) if epochs else np.empty((0, N_CHAN, N_SAMP), dtype=np.float32)


# Test su un soggetto
test_rest = load_rest_epochs(0, DATA_ROOT)
print(f'P000: {test_rest.shape} epoche riposo')  # atteso: (n, 61, 384)

In [ ]:
# ============================================================
# ESTRAZIONE FEATURE SPETTRALI (Blankertz-style)
# ============================================================

BANDS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 12),
    'SMR':   (12, 15),   # sensorimotor rhythm — chiave in Blankertz
    'beta':  (15, 30),
    'gamma': (30, 45),
}

def bandpower(epochs: np.ndarray, band: tuple, sfreq: int = SFREQ) -> np.ndarray:
    """Potenza media nella banda per ogni canale.
    Input: (n_epochs, n_chan, n_samp)
    Output: (n_chan,) — media su epoche"""
    freqs, psd = signal.welch(epochs, fs=sfreq, nperseg=SFREQ, axis=-1)  # (n, ch, freqs)
    idx = np.logical_and(freqs >= band[0], freqs <= band[1])
    return psd[:, :, idx].mean(axis=(0, 2))  # (n_chan,)


def extract_rest_features(epochs: np.ndarray) -> np.ndarray:
    """Feature vector per soggetto dal rest.
    Ritorna vettore 1D: bandpower per ogni banda × canale."""
    if len(epochs) == 0:
        return None
    feats = []
    for band_name, band in BANDS.items():
        bp = bandpower(epochs, band)  # (n_chan,)
        feats.append(np.log1p(bp))   # log-transform per stabilizzare
    return np.concatenate(feats)     # (n_bands * n_chan,)


# Test
feat_test = extract_rest_features(test_rest)
print(f'Feature vector shape: {feat_test.shape}')  # atteso: (6*61=366,)

In [ ]:
# ============================================================
# ESTRAZIONE FEATURE PER TUTTI I SOGGETTI
# ============================================================

all_subj = sorted(set(
    int(d.name.split('_')[0][1:]) for d in DATA_ROOT.iterdir() if d.is_dir()
))
print(f'Soggetti trovati: {len(all_subj)}')

features, labels, subj_ids = [], [], []

for sid in tqdm(all_subj, desc='Soggetti'):
    epochs = load_rest_epochs(sid, DATA_ROOT)
    if len(epochs) == 0:
        continue
    feat = extract_rest_features(epochs)
    if feat is None:
        continue
    features.append(feat)
    subj_ids.append(sid)
    # Label BCI literacy (solo se abbiamo bacc)
    if sid in SUBJ_BACC:
        labels.append(1 if SUBJ_BACC[sid] > ILLITERACY_THRESHOLD else 0)
    else:
        labels.append(np.nan)

X = np.array(features)          # (n_subj, n_features)
y_bacc = np.array([SUBJ_BACC.get(s, np.nan) for s in subj_ids])
y_lit  = np.array(labels)

print(f'X: {X.shape} | soggetti con bacc: {(~np.isnan(y_bacc)).sum()}')

In [ ]:
# ============================================================
# CORRELAZIONE REST FEATURE ↔ BACC
# ============================================================

mask = ~np.isnan(y_bacc)
X_labeled = X[mask]
y_labeled  = y_bacc[mask]

# Correlazione di Spearman per ogni feature
rhos, pvals = [], []
for i in range(X_labeled.shape[1]):
    rho, p = stats.spearmanr(X_labeled[:, i], y_labeled)
    rhos.append(rho)
    pvals.append(p)

rhos  = np.array(rhos)
pvals = np.array(pvals)

# Top 20 feature più correlate
top_idx = np.argsort(np.abs(rhos))[::-1][:20]
n_bands = len(BANDS)
band_names = list(BANDS.keys())

print('Top 20 feature per correlazione con bacc:')
for i in top_idx:
    band = band_names[i // N_CHAN]
    ch   = i % N_CHAN
    print(f'  [{band} ch{ch:02d}] ρ={rhos[i]:.3f}  p={pvals[i]:.4f}')

In [ ]:
# ============================================================
# CLASSIFICATORE BCI-LITERATE vs BCI-ILLITERATE (LOO-CV)
# ============================================================

lit_mask = mask & ~np.isnan(y_lit)
X_cls = X[lit_mask]
y_cls = y_lit[lit_mask].astype(int)

print(f'Literate: {y_cls.sum()}  Illiterate: {(y_cls==0).sum()}')

loo  = LeaveOneOut()
scaler = StandardScaler()
clf    = LogisticRegression(C=0.1, max_iter=1000)

y_true_all, y_prob_all = [], []
for train_idx, test_idx in loo.split(X_cls):
    X_tr = scaler.fit_transform(X_cls[train_idx])
    X_te = scaler.transform(X_cls[test_idx])
    clf.fit(X_tr, y_cls[train_idx])
    y_true_all.append(y_cls[test_idx][0])
    y_prob_all.append(clf.predict_proba(X_te)[0, 1])

auc = roc_auc_score(y_true_all, y_prob_all)
print(f'LOO-CV AUC: {auc:.3f}')

In [ ]:
# ============================================================
# PLOT: alpha/SMR power vs bacc (scatter per soggetto)
# ============================================================

# Media alpha power su elettrodi centrali (Cz, C3, C4 ≈ indici 25,24,26)
alpha_idx = 2  # banda alpha = indice 2 in BANDS
central_chs = [24, 25, 26]  # C3, Cz, C4 approssimati
alpha_central = X_labeled[:, alpha_idx * N_CHAN: (alpha_idx+1) * N_CHAN][:, central_chs].mean(axis=1)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(alpha_central, y_labeled, alpha=0.7, edgecolors='k', linewidths=0.5)
rho, p = stats.spearmanr(alpha_central, y_labeled)
ax.set_xlabel('log Alpha power (C3/Cz/C4) — rest')
ax.set_ylabel('bacc imagined speech')
ax.set_title(f'Rest alpha vs BCI performance  ρ={rho:.3f}  p={p:.4f}')
ax.axhline(ILLITERACY_THRESHOLD, color='r', linestyle='--', label='illiteracy threshold')
ax.legend()
plt.tight_layout()
plt.savefig(project_root / 'figures' / 'PB01_rest_alpha_vs_bacc.png', dpi=150)
plt.show()